# 🛡️ GRACE Đợt 3: Đối Đầu B3 (Contrastive Demonstration Selection) vs B2 (Baseline) & Ablation B3-NoDiff

**Môi Trường Thực Thi**: Kaggle GPU (T4 / P100) với Google Cloud Vertex AI MaaS  
**Hợp Đồng Đánh Giá**: `evaluation_contract.yaml`  
**Kế Hoạch Khoa Học**: `project_docs/plans/PLAN_SUPERVISOR_REQUIREMENTS_EXECUTION_20261014.md`  

### 1. Mục Tiêu Khoa Học & Nhiệm Vụ 1 (Ablation Study B3-NoDiff):
- Đánh giá đối đầu trực diện giữa **Baseline B2** (1-shot hybrid CodeT5 768-d + GraphSim) và **B3** (2-shot Contrastive Demonstration Selection: Security-Aware Retriever + Strategy C Counterexample Pair) trên bộ dữ liệu **Devign** (100 mẫu sạch, hỗ trợ tùy biến **ReVeal**).
- **Nhiệm Vụ 1 (Ablation B3-NoDiff)**: Cắt bỏ khối `--- CORE SECURITY DIFFERENCE ---` bằng cờ `--no_security_diff` để trả lời câu hỏi cốt lõi: *Hiệu năng B3 tăng vọt là nhờ Cặp mẫu cân bằng nhãn 1:1, hay nhờ khối phân tích rào chắn an ninh Core Security Difference?*
- Đo lường thống kê mẫu ghép cặp: **Kiểm định McNemar** và **Paired Bootstrap 95% Confidence Interval** (2,000 resamples).

### 2. Cấu Hình Thực Nghiệm B3 Chuẩn Figure 6 & Hợp Đồng:
- **Target Query**: Code mục tiêu ($\le 3000$ ký tự) + Đồ thị CPG Joern (tối đa 25 nodes, 25 edges).
- **1 Vulnerable Demo Reference**: Code mẫu có lỗ hổng ($\le 1200$ ký tự) + Nhãn `Vulnerable (1)`.
- **1 Safe Demo Reference**: Code mẫu an toàn tương phản ($\le 1200$ ký tự) + Nhãn `Non-vulnerable (0)`.
- **Core Security Difference**: Phân tích ranh giới an ninh cốt lõi (sanitization guards, bounds check, pointer validation).
- **Backbone LLM**: `publishers/qwen/models/qwen3-coder-480b-a35b-instruct-maas` (Location: `us-south1`, Temperature: 0.0, Seed: 42, Max tokens: 256, Retry: 5 attempts).

## Bước 1: Thiết Lập Thư Mục Làm Việc (Workspace Setup)
Tìm kiếm mã nguồn dự án trong `/kaggle/input` hoặc sao chép sang `/kaggle/working/GRACE` để có toàn quyền ghi file báo cáo và checkpoints.

In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

dest_dir = '/kaggle/working/GRACE'

# 1. Đảm bảo luôn lấy mã nguồn mới nhất từ GitHub
if not os.path.exists(dest_dir):
    print('[*] Đang clone mã nguồn GRACE mới nhất từ GitHub...')
    subprocess.run(['git', 'clone', 'https://github.com/HuuHieu03/GRACE.git', dest_dir], check=True)
else:
    print('[*] Đang cập nhật mã nguồn mới nhất từ GitHub...')
    subprocess.run(['git', '-C', dest_dir, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', dest_dir, 'reset', '--hard', 'origin/main'], check=True)

subprocess.run(['git', '-C', dest_dir, 'log', '-1', '--oneline'], check=True)

# 2. Chuyển thư mục làm việc và bổ sung sys.path
%cd /kaggle/working/GRACE
for p in ['/kaggle/working/GRACE', '/kaggle/working/GRACE/src', '/kaggle/working/GRACE/scripts']:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'✓ Thư mục làm việc hiện tại: {os.getcwd()}')
!ls -la


## Bước 2: Cài Đặt Các Gói Phụ Thuộc (Dependencies Installation)
Cài đặt các thư viện cần thiết: `google-auth`, `openai>=1.0.0`, `transformers`, `torch`, `scikit-learn`, `datasets`, `pyyaml`, `pandas`, `tqdm`.

In [ ]:
!pip install -q google-auth google-auth-oauthlib openai>=1.0.0 transformers torch scikit-learn datasets pandas tqdm pyyaml pytest
import torch
print(f'✓ PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✓ GPU Device: {torch.cuda.get_device_name(0)}')
print('✓ [Setup Complete] Các thư viện phụ thuộc đã sẵn sàng!')

## Bước 3: Xác Thực Google Cloud Vertex AI Qua Kaggle Secrets
Đọc Service Account Key từ Kaggle Secret `GCP_SA_KEY`, ghi ra file tạm `/tmp/gcp_sa_key.json`,
thiết lập `GOOGLE_APPLICATION_CREDENTIALS` và kiểm tra sinh Access Token hợp lệ qua `google.auth`.

In [ ]:
import os
import json
from pathlib import Path
from kaggle_secrets import UserSecretsClient

# 1. Đọc Kaggle Secret: GCP_SA_KEY
print('[*] Đang nạp Service Account Key từ Kaggle Secrets...')
try:
    user_secrets = UserSecretsClient()
    sa_json_str = user_secrets.get_secret('GCP_SA_KEY')
    sa_path = Path('/tmp/gcp_sa_key.json')
    with open(sa_path, 'w', encoding='utf-8') as f:
        f.write(sa_json_str.strip())
    os.chmod(sa_path, 0o600)
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(sa_path)
    print(f'✓ [OK] Đã lưu Service Account key tại: {sa_path}')
except Exception as e:
    print(f'[!] Cảnh báo nạp Kaggle Secret: {e}. Vui lòng kiểm tra Add-ons -> Secrets trên Kaggle!')

# 2. Khởi tạo cấu hình và xác thực Vertex AI
from config import config, setup_vertex_credentials
cred_file = setup_vertex_credentials()
print(f'[*] File Credentials hoạt động: {cred_file}')
print(f'[*] Vertex Project ID: {config.vertex_project_id}')
print(f'[*] Vertex Location:   {config.vertex_location}')
print(f'[*] Vertex Model:      {config.vertex_model}')

# 3. Xác thực sinh OAuth2 Token qua google.auth
try:
    import google.auth
    from google.auth.transport.requests import Request
    creds, project = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
    creds.refresh(Request())
    print(f'✓ [OK] Vertex AI Authentication Verified! Token prefix: {creds.token[:12]}... (valid={creds.valid})')
except Exception as e:
    print(f'[!] Lỗi xác thực Vertex AI: {e}')

## Bước 4: Thiết Lập Cấu Hình Bộ Dữ Liệu & Rào Chắn Hợp Đồng
Cấu hình bộ dữ liệu chạy thực nghiệm (`devign` hoặc `reveal`) và chế độ Ablation Study B3-NoDiff.
Xác nhận tập kiểm thử 100 mẫu sạch không rò rỉ mã và kho huấn luyện Train Compact CPG có đầy đủ nodes & edges theo chuẩn Joern 0.2.5 CPG.

In [ ]:
import os
import json
from pathlib import Path

# ==============================================================================
# CẤU HÌNH BỘ DỮ LIỆU ĐỢT 3: 'devign' (Mặc định cho Nhiệm vụ 1) hoặc 'reveal'
# ==============================================================================
DATASET = os.environ.get('DATASET', 'devign').lower().strip()
# Bật cờ Nhiệm Vụ 1: Ablation Study B3-NoDiff (Cắt bỏ Core Security Difference)
RUN_ABLATION_NO_DIFF = os.environ.get('RUN_ABLATION_NO_DIFF', 'true').lower().strip() in ('true', '1', 'yes')

if DATASET == 'devign':
    DATASET_NAME = 'devign'
    DATASET_DISPLAY = 'Devign'
    DEV_SPLIT_FILE = 'data/splits/devign_dev_clean_100_ids.json'
    TRAIN_CPG_FILE = 'data/processed/devign_train_compact_cpg.json.gz'
    if RUN_ABLATION_NO_DIFF:
        OUTPUT_REPORT_FILE = 'output/stage3/ablation_nodiff_comparison_report.json'
        OUTPUT_CSV_FILE = 'output/stage3/ablation_nodiff_comparison_report.csv'
    else:
        OUTPUT_REPORT_FILE = 'output/stage3/devign_b3_vs_b2_comparison_report.json'
        OUTPUT_CSV_FILE = 'output/stage3/devign_b3_vs_b2_comparison_report.csv'
else:
    DATASET_NAME = 'reveal'
    DATASET_DISPLAY = 'ReVeal'
    DEV_SPLIT_FILE = 'data/splits/reveal_dev_clean_100_ids.json'
    TRAIN_CPG_FILE = 'data/processed/reveal_train_compact_cpg.json.gz'
    if RUN_ABLATION_NO_DIFF:
        OUTPUT_REPORT_FILE = 'output/stage3/reveal_ablation_nodiff_comparison_report.json'
        OUTPUT_CSV_FILE = 'output/stage3/reveal_ablation_nodiff_comparison_report.csv'
    else:
        OUTPUT_REPORT_FILE = 'output/stage3/reveal_b3_vs_b2_comparison_report.json'
        OUTPUT_CSV_FILE = 'output/stage3/reveal_b3_vs_b2_comparison_report.csv'

print(f'[*] Bộ dữ liệu thực nghiệm: {DATASET_DISPLAY} ({DATASET_NAME}) | Ablation B3-NoDiff: {RUN_ABLATION_NO_DIFF}')
print(f'[*] Dev Split mục tiêu:      {DEV_SPLIT_FILE}')
print(f'[*] Kho Train Compact CPG:  {TRAIN_CPG_FILE}')
print(f'[*] File báo cáo đầu ra:    {OUTPUT_REPORT_FILE}')

# 1. Rào chắn hợp đồng cho tập Dev Split
split_path = Path(DEV_SPLIT_FILE)
assert split_path.exists(), f'Không tìm thấy file split: {split_path}'

with open(split_path, 'r', encoding='utf-8') as f:
    meta = json.load(f)

print(f'[*] Dataset Metadata: {meta.get("dataset")} | Split: {meta.get("split_name")}')
print(f'[*] Tổng số mẫu: {meta.get("total_samples")} (Vuln: {meta.get("vulnerable_count")}, Safe: {meta.get("safe_count")})')
print(f'[*] Seed: {meta.get("seed")} | Hash SHA-256: {meta.get("split_id_hash_sha256_short")}')

sample_details = meta.get('sample_details', [])
assert len(sample_details) == 100, f'Số mẫu chi tiết phải là 100, nhận được: {len(sample_details)}'
for idx, s in enumerate(sample_details):
    nodes = s.get('nodes', [])
    edges = s.get('edges', [])
    assert len(nodes) > 0, f'LỖI: Mẫu {s.get("id")} thiếu CPG nodes!'
    assert len(edges) > 0, f'LỖI: Mẫu {s.get("id")} thiếu CPG edges!'

# 2. Rào chắn cho kho huấn luyện Train Compact CPG
train_cpg_p = Path(TRAIN_CPG_FILE)
assert train_cpg_p.exists(), f'LỖI: Không tìm thấy file kho Train CPG tại {train_cpg_p}'
print(f'✓ [OK] Đã tìm thấy kho Train CPG: {train_cpg_p} ({train_cpg_p.stat().st_size / (1024*1024):.1f} MB)')
print(f'✓ [OK] RÀO CHẮN HỢP ĐỒNG ĐẠT CHUẨN: 100% ({len(sample_details)}/100) mẫu {DATASET_DISPLAY} có đầy đủ CPG nodes và edges.')


## Bước 5: Chạy Unit Tests Xác Minh Hệ Thống Đợt 3
Xác nhận các chốt chặn kỹ thuật: Cấu trúc prompt contrastive 2-shot, cắt bỏ security difference khi ablation, kiểm định McNemar và Paired Bootstrap.

In [ ]:
!python -m pytest tests/test_stage3_b3_vs_b2.py tests/test_stage2_contract_compliance.py tests/test_stage5_contrastive_selector.py -v
print('✓ [All Unit Tests Passed] Toàn bộ các chốt chặn Đợt 3 đã hoàn tất xuất sắc!')

## Bước 6: Thực Thi Thí Nghiệm Đợt 3 (Chạy Đối Đầu B3 vs B2 & Ablation B3-NoDiff)
Thực thi script `scripts/run_stage3_b3_vs_b2.py` với mô hình Vertex AI:
- Model: `publishers/qwen/models/qwen3-coder-480b-a35b-instruct-maas`
- Bộ dữ liệu: Tham số `--dataset` và `--dev_split` đã cấu hình ở Bước 4
- Chiến lược chọn cặp: `counterexample_pair` (Strategy C)
- Cờ `--no_security_diff`: Bật khi `RUN_ABLATION_NO_DIFF=True` để bóc tách tác động cô lập của Core Security Difference
- Báo cáo lưu tại: `OUTPUT_REPORT_FILE`

In [ ]:
import subprocess
import sys
import shutil
from pathlib import Path

print(f'[*] Khởi động thực nghiệm Đợt 3 trên {DATASET_DISPLAY} (Ablation B3-NoDiff={RUN_ABLATION_NO_DIFF})...')
cmd = [
    sys.executable, 'scripts/run_stage3_b3_vs_b2.py',
    '--dev_split', DEV_SPLIT_FILE,
    '--dataset', DATASET_NAME,
    '--model', 'publishers/qwen/models/qwen3-coder-480b-a35b-instruct-maas',
    '--provider', 'Vertex AI',
    '--location', 'us-south1',
    '--strategy', 'counterexample_pair',
    '--lambda_weight', '0.3',
    '--output', OUTPUT_REPORT_FILE,
]
if RUN_ABLATION_NO_DIFF:
    cmd.append('--no_security_diff')
print('[*] Executing:', ' '.join(cmd))
subprocess.run(cmd, check=True)

# Sao chép kết quả về thư mục gốc /kaggle/working để Kaggle nén xuất đầy đủ
out_dir = Path('/kaggle/working/output/stage3')
out_dir.mkdir(parents=True, exist_ok=True)
for f_path_str in [OUTPUT_REPORT_FILE, OUTPUT_CSV_FILE]:
    fp = Path(f_path_str)
    if fp.exists():
        shutil.copy2(fp, out_dir / fp.name)
        shutil.copy2(fp, Path('/kaggle/working') / fp.name)
        print(f'✓ [OK] Đã sao chép artifact: {fp.name}')


## Bước 7: Phân Tích & Nghiệm Thu Kết Quả Thực Nghiệm Đợt 3
Đọc file báo cáo JSON, phân tích chi tiết mức độ cải thiện của B3 so với B2 trên tất cả các chỉ số:
F1-Score, MCC, Balanced Accuracy, Precision, Recall, McNemar p-value, Bootstrap 95% CI và Báo cáo Ablation Study 3 đường cơ sở.

In [ ]:
import json
from pathlib import Path
from scripts.run_stage3_b3_vs_b2 import print_stage3_comparison_report

rep_candidates = [
    Path(OUTPUT_REPORT_FILE),
    Path(f'/kaggle/working/{OUTPUT_REPORT_FILE}'),
    Path(f'/kaggle/working/{Path(OUTPUT_REPORT_FILE).name}'),
    Path('output/stage3/ablation_nodiff_comparison_report.json'),
    Path('output/stage3/reveal_ablation_nodiff_comparison_report.json'),
    Path('output/stage3/devign_b3_vs_b2_comparison_report.json'),
    Path('output/stage3/reveal_b3_vs_b2_comparison_report.json'),
    Path('output/stage3/b3_vs_b2_comparison_report.json')
]

report_file = None
for p in rep_candidates:
    if p.exists() and p.stat().st_size > 100:
        report_file = p
        break

if report_file:
    with open(report_file, 'r', encoding='utf-8') as f:
        rep = json.load(f)
    print_stage3_comparison_report(rep)
else:
    print(f'[!] Chưa tìm thấy file báo cáo {OUTPUT_REPORT_FILE}.')
